# Fraud Detection on the PaySim Dataset (Polars)

**Goal:** build a machine-learning pipeline that detects fraudulent mobile-money transactions, using **Polars** for all data handling.

**Pipeline**
1. Import libraries & load data
2. Data understanding & quality checks
3. Exploratory data analysis (EDA)
4. Feature engineering
5. Train/test split & class imbalance
6. Model training (Logistic Regression, Random Forest, Gradient Boosting)
7. Evaluation (Precision, Recall, F1, ROC-AUC, PR-AUC, confusion matrix)
8. Threshold tuning & feature importance
9. Fraud **detection** on new transactions (+ saving the model)

**Dataset:** PaySim synthetic mobile-money transactions (Kaggle: `ealaxi/paysim1`, ~6.36M rows).
Columns: `step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig, nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud`.

## 1. Import libraries

In [ ]:
import os, warnings, joblib
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, precision_score, recall_score, f1_score)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(20)
RANDOM_STATE = 42

## 2. Load the data

Option A (default): download automatically with `kagglehub` (`pip install kagglehub`, needs Kaggle access).
Option B: set `DATA_PATH` to a local copy of `PS_20174392719_1491204439457_log.csv`.

`SAMPLE_FRAC` lets you work with a fraction of the data for fast experiments (set to `1.0` for the full dataset).

In [ ]:
DATA_PATH = os.environ.get("PAYSIM_CSV", "")   # e.g. "PS_20174392719_1491204439457_log.csv"
SAMPLE_FRAC = 1.0                              # use 0.2 for a quicker run

if not DATA_PATH:
    import kagglehub
    folder = kagglehub.dataset_download("ealaxi/paysim1")
    DATA_PATH = os.path.join(folder, "PS_20174392719_1491204439457_log.csv")

df = pl.read_csv(DATA_PATH)
if SAMPLE_FRAC < 1.0:
    df = df.sample(fraction=SAMPLE_FRAC, seed=RANDOM_STATE)

# stable row id so we can map model results back to the original rows later
df = df.with_row_index("row_id")

print("Shape:", df.shape)
df.head()

## 3. Data understanding & quality checks

In [ ]:
print(df.schema)
df.glimpse()

In [ ]:
display(df.drop("row_id").describe())
print("Null counts:")
display(df.null_count())
print("Duplicate rows (excluding row_id):", int(df.drop("row_id").is_duplicated().sum()))

In [ ]:
class_counts = df["isFraud"].value_counts().sort("isFraud")
display(class_counts)
print(f"Fraud rate: {df['isFraud'].mean()*100:.4f}%  ->  highly imbalanced")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["Legit", "Fraud"], class_counts["count"].to_list(), color=["steelblue", "crimson"])
ax.set_yscale("log"); ax.set_title("Class distribution (log scale)")
plt.show()

## 4. Exploratory data analysis

### 4.1 Fraud by transaction type

In [ ]:
type_summary = (df.group_by("type")
                  .agg(pl.len().alias("transactions"),
                       pl.col("isFraud").sum().alias("frauds"),
                       pl.col("isFraud").mean().alias("fraud_rate"))
                  .sort("type"))
display(type_summary)

types = type_summary["type"].to_list()
legit_n = (type_summary["transactions"] - type_summary["frauds"]).to_numpy()
fraud_n = type_summary["frauds"].to_numpy()
x = np.arange(len(types))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(x - 0.2, legit_n, 0.4, label="Legit"); axes[0].bar(x + 0.2, fraud_n, 0.4, label="Fraud", color="crimson")
axes[0].set_xticks(x, types); axes[0].set_yscale("log"); axes[0].legend(); axes[0].set_title("Transactions by type and label (log)")
axes[1].bar(types, type_summary["fraud_rate"].to_numpy(), color="crimson"); axes[1].set_title("Fraud rate by type")
plt.tight_layout(); plt.show()

**Observation:** in PaySim, fraud occurs only in `TRANSFER` and `CASH_OUT` transactions. We use this to focus the model.

### 4.2 Transaction amount

In [ ]:
sub = df.filter(pl.col("type").is_in(["TRANSFER", "CASH_OUT"]))
amt_legit = sub.filter(pl.col("isFraud") == 0)["amount"].to_numpy()
amt_fraud = sub.filter(pl.col("isFraud") == 1)["amount"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot([amt_legit, amt_fraud], labels=["Legit", "Fraud"]); axes[0].set_yscale("log")
axes[0].set_title("Amount by class (TRANSFER / CASH_OUT, log)")
bins = np.linspace(0, np.log1p(max(amt_legit.max(), amt_fraud.max())), 60)
axes[1].hist(np.log1p(amt_legit), bins=bins, density=True, alpha=0.6, label="Legit")
axes[1].hist(np.log1p(amt_fraud), bins=bins, density=True, alpha=0.6, label="Fraud", color="crimson")
axes[1].set_title("log(1+amount) distribution"); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.3 Fraud over time (`step` = 1 hour of simulation)

In [ ]:
by_hour = (df.with_columns((pl.col("step") % 24).alias("hour"))
             .group_by("hour").agg(pl.col("isFraud").sum().alias("frauds")).sort("hour"))
by_step = df.group_by("step").agg(pl.col("isFraud").sum().alias("frauds")).sort("step")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(by_hour["hour"].to_numpy(), by_hour["frauds"].to_numpy(), color="crimson"); axes[0].set_title("Fraud count by hour of day")
axes[1].plot(by_step["step"].to_numpy(), by_step["frauds"].to_numpy()); axes[1].set_title("Fraud count by step")
plt.tight_layout(); plt.show()

### 4.4 Account balance behaviour

In [ ]:
fraud = df.filter(pl.col("isFraud") == 1)
legit_tc = df.filter((pl.col("isFraud") == 0) & pl.col("type").is_in(["TRANSFER", "CASH_OUT"]))

print(f"Fraud: origin balance emptied (newbalanceOrig == 0): {(fraud['newbalanceOrig'] == 0).mean()*100:.1f}%")
print(f"Legit TRANSFER/CASH_OUT: origin balance emptied:      {(legit_tc['newbalanceOrig'] == 0).mean()*100:.1f}%")
print(f"Fraud: destination old balance == 0: {(fraud['oldbalanceDest'] == 0).mean()*100:.1f}%")
print("\nBuilt-in rule isFlaggedFraud flags", int(df['isFlaggedFraud'].sum()),
      "transactions vs", int(df['isFraud'].sum()), "real frauds")

In [ ]:
num_cols = ["step", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "isFraud"]
corr = np.corrcoef(df.select(num_cols).to_numpy().astype(float).T)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)), num_cols, rotation=45, ha="right"); ax.set_yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Correlation matrix"); plt.show()

## 5. Feature engineering

* **errorBalanceOrig** = `newbalanceOrig + amount - oldbalanceOrg` (0 for a consistent debit)
* **errorBalanceDest** = `oldbalanceDest + amount - newbalanceDest` (0 for a consistent credit)
* **origEmptied / destZeroBefore** flags (typical fraud: drain the account, send to an empty account)
* **isMerchantDest** (destination ids starting with `M`), **hour** of day, **amount_to_balance** ratio

Only `TRANSFER` and `CASH_OUT` can be fraudulent in this data, so we model that subset.

In [ ]:
FEATURES = ["step", "hour", "type_TRANSFER", "amount",
            "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
            "errorBalanceOrig", "errorBalanceDest", "origEmptied", "destZeroBefore",
            "isMerchantDest", "amount_to_balance"]
TARGET = "isFraud"

def add_features(d: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered columns to a PaySim-style frame (polars expressions)."""
    return d.with_columns(
        (pl.col("step") % 24).alias("hour"),
        (pl.col("type") == "TRANSFER").cast(pl.Int8).alias("type_TRANSFER"),
        (pl.col("newbalanceOrig") + pl.col("amount") - pl.col("oldbalanceOrg")).alias("errorBalanceOrig"),
        (pl.col("oldbalanceDest") + pl.col("amount") - pl.col("newbalanceDest")).alias("errorBalanceDest"),
        ((pl.col("oldbalanceOrg") > 0) & (pl.col("newbalanceOrig") == 0)).cast(pl.Int8).alias("origEmptied"),
        ((pl.col("oldbalanceDest") == 0) & (pl.col("newbalanceDest") == 0)).cast(pl.Int8).alias("destZeroBefore"),
        pl.col("nameDest").str.starts_with("M").cast(pl.Int8).alias("isMerchantDest"),
        (pl.col("amount") / (pl.col("oldbalanceOrg") + 1)).alias("amount_to_balance"),
    )

data = add_features(df.filter(pl.col("type").is_in(["TRANSFER", "CASH_OUT"])))
print("Modelling subset:", data.shape, "| fraud rate: %.4f%%" % (data[TARGET].mean() * 100))
data.select(FEATURES).head()

## 6. Train/test split & class imbalance
Stratified split so both sets keep the same (tiny) fraud rate. Instead of discarding data, we use **class weights** so errors on the rare fraud class cost more. Accuracy is misleading here, so we focus on **Precision, Recall, F1 and PR-AUC**.

In [ ]:
X = data.select(FEATURES).to_numpy().astype(np.float64)
y = data[TARGET].to_numpy()
row_ids = data["row_id"].to_numpy()

idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = X[idx_train], X[idx_test], y[idx_train], y[idx_test]
test_row_ids = row_ids[idx_test]

print("Train:", X_train.shape, "| fraud:", int(y_train.sum()))
print("Test :", X_test.shape,  "| fraud:", int(y_test.sum()))

## 7. Model training

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=14, min_samples_leaf=3, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.1, class_weight="balanced", random_state=RANDOM_STATE),
}

fitted, probas = {}, {}
for name, model in models.items():
    print(f"Training {name} ...")
    model.fit(X_train, y_train)
    fitted[name] = model
    probas[name] = model.predict_proba(X_test)[:, 1]
print("Done.")

## 8. Evaluation

In [ ]:
rows = []
for name, p in probas.items():
    pred = (p >= 0.5).astype(int)
    rows.append({"Model": name,
                 "Precision": precision_score(y_test, pred, zero_division=0),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, p),
                 "PR-AUC": average_precision_score(y_test, p)})
results = pl.DataFrame(rows).sort("PR-AUC", descending=True)
best_name = results["Model"][0]
display(results.with_columns(pl.col(pl.Float64).round(4)))
print("Best model by PR-AUC:", best_name)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.4f})")
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, p):.4f})")
axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set(title="ROC curve", xlabel="FPR", ylabel="TPR"); axes[0].legend()
axes[1].set(title="Precision-Recall curve", xlabel="Recall", ylabel="Precision"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(probas), figsize=(5 * len(probas), 4))
for ax, (name, p) in zip(np.atleast_1d(axes), probas.items()):
    cm = confusion_matrix(y_test, (p >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name)
plt.tight_layout(); plt.show()

print(classification_report(y_test, (probas[best_name] >= 0.5).astype(int), target_names=["Legit", "Fraud"], digits=4))

## 9. Threshold tuning
The default 0.5 threshold is rarely optimal for fraud. Here we pick the threshold that maximises F1; if missing fraud is costlier than a false alarm, choose the threshold that reaches a required recall instead.

In [ ]:
best_model = fitted[best_name]
p = probas[best_name]
prec, rec, thr = precision_recall_curve(y_test, p)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
BEST_THR = float(thr[np.argmax(f1)])
print(f"Threshold maximising F1: {BEST_THR:.4f}  (F1={f1.max():.4f})")

plt.figure(figsize=(8, 4))
plt.plot(thr, prec[:-1], label="Precision"); plt.plot(thr, rec[:-1], label="Recall"); plt.plot(thr, f1, label="F1")
plt.axvline(BEST_THR, color="k", ls="--"); plt.xlabel("Threshold"); plt.legend(); plt.title("Threshold trade-off")
plt.show()

pred_tuned = (p >= BEST_THR).astype(int)
print(classification_report(y_test, pred_tuned, target_names=["Legit", "Fraud"], digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_tuned), display_labels=["Legit", "Fraud"]).plot(cmap="Blues", values_format="d")
plt.title(f"{best_name} @ threshold {BEST_THR:.3f}"); plt.show()

## 10. Feature importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = np.asarray(best_model.feature_importances_)
else:
    # permutation importance on a test subsample (works for any model, incl. HistGradientBoosting / pipelines)
    sel = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=min(50000, len(X_test)), replace=False)
    pi = permutation_importance(best_model, X_test[sel], y_test[sel], scoring="average_precision",
                                n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    imp = pi.importances_mean

imp_df = pl.DataFrame({"feature": FEATURES, "importance": imp}).sort("importance")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"].to_list(), imp_df["importance"].to_numpy(), color="teal")
ax.set_title(f"Feature importance - {best_name}"); plt.show()

## 11. Fraud detection on transactions
We now use the trained model for actual **detection**: score transactions, flag suspicious ones, and compare against the dataset's built-in `isFlaggedFraud` rule.

In [ ]:
def build_features(raw: pl.DataFrame) -> np.ndarray:
    """Raw PaySim-style transactions -> model feature matrix."""
    return add_features(raw).select(FEATURES).to_numpy().astype(np.float64)

def detect_fraud(raw: pl.DataFrame, model=best_model, threshold=BEST_THR) -> pl.DataFrame:
    """Score transactions; returns the input with fraud_probability and is_fraud_predicted columns.
    Types other than TRANSFER / CASH_OUT are never fraud in PaySim, so they get probability 0."""
    mask = raw["type"].is_in(["TRANSFER", "CASH_OUT"]).to_numpy()
    prob = np.zeros(raw.height)
    if mask.any():
        prob[mask] = model.predict_proba(build_features(raw.filter(pl.Series(mask))))[:, 1]
    return (raw.with_columns(pl.Series("fraud_probability", prob))
               .with_columns((pl.col("fraud_probability") >= threshold).cast(pl.Int8).alias("is_fraud_predicted")))

In [ ]:
# 11.1 Batch detection on the held-out rows of the original data (all transaction types)
holdout = df.filter(pl.col("row_id").is_in(test_row_ids))
scored = detect_fraud(holdout)

flagged = scored.filter(pl.col("is_fraud_predicted") == 1).sort("fraud_probability", descending=True)
print(f"Scored: {scored.height:,} | flagged as fraud: {flagged.height:,} | actual frauds: {int(scored['isFraud'].sum()):,}")
flagged.select(["step", "type", "amount", "nameOrig", "nameDest", "oldbalanceOrg", "newbalanceOrig",
                "fraud_probability", "isFraud"]).head(10)

In [ ]:
# 11.2 Compare with the dataset's built-in rule (isFlaggedFraud)
yt = scored["isFraud"].to_numpy()
for label, col in [("Built-in rule", "isFlaggedFraud"), ("ML model", "is_fraud_predicted")]:
    yp = scored[col].to_numpy()
    print(f"{label:14s} -> recall: {recall_score(yt, yp):.4f} | precision: {precision_score(yt, yp, zero_division=0):.4f}")

In [ ]:
# 11.3 Detect a single new transaction (real-time style)
new_txn = pl.DataFrame({
    "step": [300], "type": ["TRANSFER"], "amount": [181000.0],
    "nameOrig": ["C1000000001"], "oldbalanceOrg": [181000.0], "newbalanceOrig": [0.0],
    "nameDest": ["C2000000002"], "oldbalanceDest": [0.0], "newbalanceDest": [0.0]})

res = detect_fraud(new_txn)
print(f"Fraud probability: {res['fraud_probability'][0]:.4f}")
print("Decision:", "FRAUD - block / review" if res["is_fraud_predicted"][0] else "Legitimate")

## 12. Save the model

In [ ]:
joblib.dump({"model": best_model, "features": FEATURES, "threshold": BEST_THR, "name": best_name},
            "fraud_detection_model.joblib")
print("Saved -> fraud_detection_model.joblib")
# Reload: art = joblib.load("fraud_detection_model.joblib"); art["model"].predict_proba(X)

---
# Part 2 - Improving & validating the model

The baseline works, so the next step is to make it **more reliable**:
1. Compare imbalance-handling strategies
2. Tune hyper-parameters (scored on PR-AUC)
3. Pick the decision threshold on a **validation set** (not on the test set) and re-evaluate
4. Check robustness with a **time-based split**
5. Translate results into **business impact** (money caught vs. missed)
6. Re-run detection with the final model and save it

## 13. Compare imbalance-handling strategies
Same model (Gradient Boosting) trained three ways: no handling, class weights, and random undersampling of legitimate transactions (10:1). Because undersampling shifts probabilities, compare **PR-AUC** and **best F1** (threshold-independent) rather than only F1 at 0.5.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

def best_threshold_f1(y_true, p):
    pr, rc, th = precision_recall_curve(y_true, p)
    f = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-12)
    return float(th[np.argmax(f)]), float(f.max())

rng = np.random.RandomState(RANDOM_STATE)
fraud_idx = np.where(y_train == 1)[0]
legit_idx = np.where(y_train == 0)[0]
under_idx = np.concatenate([fraud_idx,
                            rng.choice(legit_idx, size=min(len(legit_idx), 10 * len(fraud_idx)), replace=False)])

strategies = {
    "No handling":        (X_train, y_train, None),
    "Class weights":      (X_train, y_train, "balanced"),
    "Undersample (10:1)": (X_train[under_idx], y_train[under_idx], None),
}
rows = []
for name, (Xs, ys, cw) in strategies.items():
    m = HistGradientBoostingClassifier(max_iter=200, class_weight=cw, random_state=RANDOM_STATE).fit(Xs, ys)
    p_s = m.predict_proba(X_test)[:, 1]
    rows.append({"Strategy": name,
                 "PR-AUC": average_precision_score(y_test, p_s),
                 "ROC-AUC": roc_auc_score(y_test, p_s),
                 "F1 @0.5": f1_score(y_test, (p_s >= 0.5).astype(int)),
                 "Best F1": best_threshold_f1(y_test, p_s)[1]})
imbalance_results = pl.DataFrame(rows).sort("PR-AUC", descending=True)
display(imbalance_results.with_columns(pl.col(pl.Float64).round(4)))

## 14. Hyper-parameter tuning
Randomized search with 3-fold stratified CV, scored by **average precision (PR-AUC)**. For speed the search runs on a stratified subsample of the training data (`SEARCH_N` rows); the final model is then trained on the full training set.

In [ ]:
SEARCH_N = 400_000
if len(y_train) > SEARCH_N:
    sub_idx, _ = train_test_split(np.arange(len(y_train)), train_size=SEARCH_N, stratify=y_train, random_state=RANDOM_STATE)
else:
    sub_idx = np.arange(len(y_train))

param_dist = {
    "learning_rate":     [0.03, 0.05, 0.1, 0.2],
    "max_iter":          [100, 200, 300],
    "max_depth":         [None, 6, 10],
    "max_leaf_nodes":    [15, 31, 63],
    "min_samples_leaf":  [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 1.0],
}
search = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    param_distributions=param_dist, n_iter=8, scoring="average_precision",
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1, random_state=RANDOM_STATE, refit=False, verbose=1)
search.fit(X_train[sub_idx], y_train[sub_idx])

cv = search.cv_results_
search_table = pl.DataFrame([{"rank": int(r), "cv_PR-AUC": float(m), "std": float(s), **{k: str(v) for k, v in prm.items()}}
                             for r, m, s, prm in zip(cv["rank_test_score"], cv["mean_test_score"], cv["std_test_score"], cv["params"])]
                            ).sort("rank")
display(search_table.with_columns(pl.col(pl.Float64).round(4)))
print("Best params:", search.best_params_)

## 15. Final model, validation-based threshold & honest test evaluation
* Hold out 20% of the **training** data as a validation set and choose the threshold there (the baseline in section 9 chose its threshold on the test set, which is slightly optimistic).
* Evaluate once on the untouched test set.

In [ ]:
idx_fit, idx_val = train_test_split(np.arange(len(y_train)), test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

final_model = HistGradientBoostingClassifier(**search.best_params_, class_weight="balanced", random_state=RANDOM_STATE)
final_model.fit(X_train[idx_fit], y_train[idx_fit])

FINAL_THR, val_f1 = best_threshold_f1(y_train[idx_val], final_model.predict_proba(X_train[idx_val])[:, 1])
print(f"Threshold chosen on validation set: {FINAL_THR:.4f} (validation F1={val_f1:.4f})")

p_final = final_model.predict_proba(X_test)[:, 1]
pred_final = (p_final >= FINAL_THR).astype(int)

def summarize(label, y_true, p, thr):
    yp = (p >= thr).astype(int)
    return {"Model": label, "Threshold": thr,
            "Precision": precision_score(y_true, yp, zero_division=0), "Recall": recall_score(y_true, yp),
            "F1": f1_score(y_true, yp), "ROC-AUC": roc_auc_score(y_true, p), "PR-AUC": average_precision_score(y_true, p)}

compare = pl.DataFrame([summarize(f"Baseline ({best_name})", y_test, probas[best_name], BEST_THR),
                        summarize("Tuned Gradient Boosting", y_test, p_final, FINAL_THR)])
display(compare.with_columns(pl.col(pl.Float64).round(4)))
print(classification_report(y_test, pred_final, target_names=["Legit", "Fraud"], digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final), display_labels=["Legit", "Fraud"]).plot(cmap="Blues", values_format="d")
plt.title("Tuned model - test set"); plt.show()

## 16. Robustness check: time-based split
A random split lets the model see 'future' transactions during training. Here we train on the earliest 70% of simulation steps and test on the latest 30%. The `step` column is excluded so the model cannot simply memorise time.

In [ ]:
step_col = X[:, FEATURES.index("step")]
cut = np.quantile(step_col, 0.7)
tr_mask, te_mask = step_col <= cut, step_col > cut
cols_no_step = [i for i, f in enumerate(FEATURES) if f != "step"]

m_time = HistGradientBoostingClassifier(**search.best_params_, class_weight="balanced", random_state=RANDOM_STATE)
m_time.fit(X[tr_mask][:, cols_no_step], y[tr_mask])
p_time = m_time.predict_proba(X[te_mask][:, cols_no_step])[:, 1]

print(f"Train steps <= {cut:.0f} ({tr_mask.sum():,} rows, {int(y[tr_mask].sum())} frauds) | "
      f"Test steps > {cut:.0f} ({te_mask.sum():,} rows, {int(y[te_mask].sum())} frauds)")
display(pl.DataFrame([summarize("Time-based split", y[te_mask], p_time, FINAL_THR)]).with_columns(pl.col(pl.Float64).round(4)))

If the time-split scores stay close to the random-split scores, the model generalises forward in time rather than memorising.

## 17. Business impact
Accuracy metrics hide what matters to a business: **how much money** is stopped, and what the false alarms cost. `FALSE_ALARM_COST` is an assumption - replace it with your real review cost per flagged transaction.

In [ ]:
FALSE_ALARM_COST = 10.0   # assumed cost of manually reviewing one flagged transaction

amount_test = X_test[:, FEATURES.index("amount")]
is_fraud, flagged_mask = (y_test == 1), (pred_final == 1)

fraud_total   = amount_test[is_fraud].sum()
fraud_caught  = amount_test[is_fraud & flagged_mask].sum()
fraud_missed  = fraud_total - fraud_caught
false_alarms  = int((~is_fraud & flagged_mask).sum())
review_cost   = false_alarms * FALSE_ALARM_COST

print(f"Total fraud value in test set : {fraud_total:>18,.0f}")
print(f"Caught by the model           : {fraud_caught:>18,.0f}  ({fraud_caught / fraud_total * 100:.1f}%)")
print(f"Missed                        : {fraud_missed:>18,.0f}")
print(f"False alarms                  : {false_alarms:>18,d}  -> review cost ~ {review_cost:,.0f}")
print(f"Net benefit (caught - review) : {fraud_caught - review_cost:>18,.0f}")

## 18. Final detection & comparison of the three systems

In [ ]:
scored_final = detect_fraud(holdout, model=final_model, threshold=FINAL_THR)

yt = scored_final["isFraud"].to_numpy()
systems = {
    "Built-in rule (isFlaggedFraud)": scored_final["isFlaggedFraud"].to_numpy(),
    "Baseline ML model":              scored["is_fraud_predicted"].to_numpy(),
    "Tuned ML model":                 scored_final["is_fraud_predicted"].to_numpy(),
}
display(pl.DataFrame([{"System": k, "Flagged": int(v.sum()),
                       "Precision": precision_score(yt, v, zero_division=0),
                       "Recall": recall_score(yt, v), "F1": f1_score(yt, v)} for k, v in systems.items()]
                     ).with_columns(pl.col(pl.Float64).round(4)))

res = detect_fraud(new_txn, model=final_model, threshold=FINAL_THR)
print(f"New transaction -> fraud probability {res['fraud_probability'][0]:.4f} | "
      + ("FRAUD - block / review" if res["is_fraud_predicted"][0] else "Legitimate"))

joblib.dump({"model": final_model, "features": FEATURES, "threshold": FINAL_THR,
             "params": search.best_params_, "name": "Tuned HistGradientBoosting"}, "fraud_detection_model_tuned.joblib")
print("Saved -> fraud_detection_model_tuned.joblib")

## 19. Conclusions & next steps

* Fraud is extremely rare (~0.13% of all transactions) and occurs **only in `TRANSFER` and `CASH_OUT`**.
* Balance-consistency features (`errorBalanceOrig`, `errorBalanceDest`, `origEmptied`, `destZeroBefore`) are the strongest signals.
* Tree-based models beat Logistic Regression; PR-AUC and recall at a tuned threshold are the right yardsticks, not accuracy.
* Part 2 tightened the evaluation: imbalance strategies were compared, hyper-parameters tuned on PR-AUC, the threshold chosen on a validation set, and robustness checked with a time-based split.
* Translating results into money caught vs. review cost gives a decision-ready view of the detector.

**Caveats:** PaySim is synthetic, so scores are optimistic compared with real data. Balance features may not be available at decision time in production (leakage risk) - validate this. `FALSE_ALARM_COST` is an assumption.

**Further ideas:** XGBoost/LightGBM and SMOTE comparisons, SHAP explanations per flagged transaction, anomaly detection (Isolation Forest / autoencoder) for unseen fraud patterns, Polars `scan_csv` lazy pipelines for scale, and serving the model via FastAPI/Streamlit with drift monitoring.